In [1]:
# Instale as bibliotecas necessárias no seu ambiente
!pip install mlflow sagemaker xgboost scikit-learn -Uq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
autogluon-core 1.5.0 requires scikit-learn<1.8.0,>=1.4.0, but you have scikit-learn 1.9.0 which is incompatible.
autogluon-features 1.5.0 requires scikit-learn<1.8.0,>=1.4.0, but you have scikit-learn 1.9.0 which is incompatible.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.3.0 which is incompatible.
autogluon-multimodal 1.5.0 requires scikit-learn<1.8.0,>=1.4.0, but you have scikit-learn 1.9.0 which is incompatible.
autogluon-tabular 1.5.0 requ

In [2]:
import mlflow
import boto3

# Substitua pelo ARN do seu MLflow App criado no console do SageMaker
tracking_server_arn = "arn:aws:sagemaker:us-east-2:906513713169:mlflow-app/app-G677ZQHBYJVJ"

# Configura o MLflow para enviar os logs para o SageMaker
mlflow.set_tracking_uri(tracking_server_arn)

print(f"Conectado ao Tracking Server: {mlflow.get_tracking_uri()}")

mlflow.autolog()

2026/07/09 13:53:32 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.


2026/07/09 13:53:32 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.


Conectado ao Tracking Server: arn:aws:sagemaker:us-east-2:906513713169:mlflow-app/app-G677ZQHBYJVJ


In [4]:
import pandas as pd
import numpy as np
import os
import sklearn
import mlflow
import mlflow.lightgbm
import warnings
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, classification_report, f1_score
from lightgbm import LGBMClassifier

# Silencia avisos de depreciação internos para limpar a saída do console
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Configuração para o scikit-learn devolver DataFrames estruturados
sklearn.set_config(transform_output="pandas")

# ==============================================================================
# 1. TRANSFORMERS CUSTOMIZADOS (ENGENHARIA E PREPARAÇÃO)
# ==============================================================================

class ImputadorDistribuicao(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        if 'dias_desde_ultimo_sinistro' in X.columns:
            valores_validos = X['dias_desde_ultimo_sinistro'].replace(-1, np.nan).dropna()
            self.distribuicao_treino_ = valores_validos.values if len(valores_validos) > 0 else np.array([0])
        else:
            self.distribuicao_treino_ = np.array([0])
        return self

    def transform(self, X):
        X = X.copy()
        if 'dias_desde_ultimo_sinistro' in X.columns:
            X['dias_desde_ultimo_sinistro'] = X['dias_desde_ultimo_sinistro'].replace(-1, np.nan)
            nulos_mask = X['dias_desde_ultimo_sinistro'].isna()
            num_nulos = nulos_mask.sum()

            if num_nulos > 0:
                amostra = np.random.choice(self.distribuicao_treino_, size=num_nulos, replace=True)
                X.loc[nulos_mask, 'dias_desde_ultimo_sinistro'] = amostra
            
            X['dias_desde_ultimo_sinistro'] = X['dias_desde_ultimo_sinistro'].fillna(-1)
        return X

        
class EngenhariaDeFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.media_desc_ = X["desconto_aplicado_pct"].mean() if "desconto_aplicado_pct" in X.columns else 0
        return self

    def transform(self, X):
        X = X.copy()
        
        premio = X.get('valor_premio_anual', 0).fillna(0)
        cobertura = X.get('valor_cobertura_total', 1).fillna(1).replace(0, 1)
        renda = X.get('renda_anual', 1).fillna(1).replace(0, 1)
        apolices = X.get('num_apolices_ativas', 1).fillna(1).replace(0, 1)
        km = X.get('km_anual_estimado', 1).fillna(1).replace(0, 1)
        
        X['custo_beneficio'] = premio / cobertura
        
        if 'metodo_pagamento' in X.columns and 'pagamento_em_dia' in X.columns:
            X['friccao_pagamento'] = ((X['metodo_pagamento'].astype(str).str.lower() == 'boleto') & (X['pagamento_em_dia'] == 0)).astype(int)
        else:
            X['friccao_pagamento'] = 0
            
        X["cliente_novo_alto_desconto"] = ((X.get("renovacoes_consecutivas", 0).fillna(0) <= 1) & (X.get("desconto_aplicado_pct", 0).fillna(0) > self.media_desc_)).astype(int)

        X['reclamacoes_s_resposta'] = ((X.get('num_reclamacoes_12m', 0).fillna(0) > 0) & (X.get('dias_ultimo_contato', 0).fillna(0) > 90) & (X.get('satisfacao_nps', 10).fillna(10) <= 6)).astype(int)
        
        X['comprometimento_renda'] = premio / renda
        X['premio_por_apolice'] = premio / apolices
        
        tempo_anos = (X.get('tempo_cliente_dias', 365).fillna(365) / 365).replace(0, 0.1)
        X['frequencia_sinistros_tempo'] = X.get('num_sinistros_historico', 0).fillna(0) / tempo_anos
        
        X['isolamento_digital'] = ((X.get('nunca_logou', 0).fillna(0) == 1) | (X.get('ultimo_login_portal_dias', 0).fillna(0) > 180)).astype(int)
        X['renda_per_capita'] = X.get('renda_anual', 0).fillna(0) / (X.get('qtd_dependentes', 0).fillna(0) + 1)
        X['custo_por_km'] = premio / km
        X['peso_franquia_premio'] = X.get('franquia_media', 0).fillna(0) / premio.replace(0, 1)
        
        X['idade_ingresso'] = X.get('idade', 30).fillna(30) - (X.get('tempo_cliente_dias', 0).fillna(0) / 365)

        # Regras de Negócio baseadas no Insight Gráfico de Faixa Etária (Idade)
        X['jovem_baixa_renda'] = ((X.get('idade', 30).fillna(30) <= 35) & (X['comprometimento_renda'] > 0.05)).astype(int)
        X['senior_com_sinistro'] = ((X.get('idade', 30).fillna(30) >= 56) & (X.get('teve_sinistro', 0).fillna(0) == 1)).astype(int)

        return X


class CriadorFaixaEtaria(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if 'idade' in X.columns:
            idade_preenchida = X['idade'].fillna(30)
            bins = [0, 25, 35, 45, 55, 65, np.inf]
            labels = ['Até 25', '26-35', '36-45', '46-55', '56-65', 'Mais de 65']
            X['faixa_etaria'] = pd.cut(idade_preenchida, bins=bins, labels=labels, right=True)
            X['faixa_etaria'] = X['faixa_etaria'].astype(str)
        return X


class CodificadorOrdinalManual(BaseEstimator, TransformerMixin):
    def __init__(self, mapping_dicts):
        self.mapping_dicts = mapping_dicts

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col, map_dict in self.mapping_dicts.items():
            if col in X.columns:
                X[col] = X[col].map(map_dict)
                X[col] = X[col].fillna(-1)
        return X


class RemovedorDeColunas(BaseEstimator, TransformerMixin):
    def __init__(self, colunas):
        self.colunas = colunas

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=[c for c in self.colunas if c in X.columns], errors='ignore')


# ==============================================================================
# 2. FUNÇÃO CENTRAL DE CONSTRUÇÃO DO DATA PIPELINE
# ==============================================================================

def build_pipeline(
    data_dir: str = "../bases_tratadas",
    target_col: str = "churned",
    val_size: float = 0.15,
    test_size: float = 0.15,
    random_state: int = 42,
) -> dict:
    
    if not os.path.exists(data_dir):
        print(f"⚠️ Diretório '{data_dir}' não encontrado. Buscando arquivos na pasta raiz atual ('.')...")
        data_dir = "."

    # Leitura dos arquivos individuais usando a nomenclatura correta
    df_cadastros = pd.read_csv(f'{data_dir}/cadastro_tratado.csv')
    df_sinistros = pd.read_csv(f'{data_dir}/sinistros_tratado.csv')
    df_mkt       = pd.read_csv(f'{data_dir}/engajamento_marketing_tratado.csv')
    df_contratos = pd.read_csv(f'{data_dir}/contratos_apolices_tratado.csv')
    
    try:
        df_churn = pd.read_csv(f'{data_dir}/churn_.csv')
    except FileNotFoundError:
        df_churn = df_cadastros[['id_cliente', target_col]].copy()

    for df in [df_cadastros, df_sinistros, df_churn, df_contratos, df_mkt]:
        df['id_cliente'] = df['id_cliente'].astype(str).str.strip()

    # União das bases pelo ID do cliente
    df = (
        df_churn
        .merge(df_contratos, on='id_cliente', how='left')
        .merge(df_mkt,       on='id_cliente', how='left')
        .merge(df_cadastros, on='id_cliente', how='left')
        .merge(df_sinistros, on='id_cliente', how='left')
        .copy()
    )

    df['nunca_logou'] = df['nunca_logou'].fillna(1)
    df['teve_sinistro'] = df['teve_sinistro'].fillna(0)

    # Divisão estratificada das partições
    train_df, temp_df = train_test_split(
        df,
        test_size=(val_size + test_size),
        stratify=df[target_col],
        random_state=random_state,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=test_size / (val_size + test_size),
        stratify=temp_df[target_col],
        random_state=random_state,
    )

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]
    X_val   = val_df.drop(columns=[target_col])
    y_val   = val_df[target_col]
    X_test  = test_df.drop(columns=[target_col])
    y_test  = test_df[target_col]

    encoding_maps = {
        "tipo_cobertura":      {"Básica": 1, "Padrão": 2, "Premium": 3},
        "segmento_marketing":  {"Bronze": 1, "Prata": 2, "Ouro": 3, "Diamante": 4},
        "escolaridade":        {"Fundamental": 1, "Medio": 2, "Superior": 3, "Pos": 4},
        "faixa_etaria":        {"Até 25": 1, "26-35": 2, "36-45": 3, "46-55": 4, "56-65": 5, "Mais de 65": 6}
    }

    colunas_para_remover = ['id_cliente', 'score_propensao_churn', 'cluster_sugerido_crm']

    # Mapeamento completo e explícito das variáveis categóricas (Remove erros com 'carta' ou 'whatsapp')
    one_hot_cols = [
        'estado_civil', 'genero', 'canal_aquisicao', 'metodo_pagamento', 
        'regiao_vendas', 'tipo_veiculo', 'canal_preferencial_contato'
    ]

    numeric_cols = [
        'idade', 'tempo_cliente_dias', 'qtd_dependentes', 'renda_anual', 
        'valor_premio_anual', 'valor_cobertura_total', 'desconto_aplicado_pct', 
        'franquia_media', 'num_apolices_ativas', 'num_reclamacoes_12m', 
        'num_sinistros_historico', 'num_ligacoes_suporte_12m', 'num_acessos_app_mes', 
        'ultimo_login_portal_dias', 'dias_ultimo_contato', 'satisfacao_nps',
        'score_engajamento_digital', 'indice_relacionamento', 'ano_veiculo', 'km_anual_estimado'
    ]
    
    features_engenharia = [
        'custo_beneficio', 'friccao_pagamento', 'cliente_novo_alto_desconto',
        'reclamacoes_s_resposta', 'comprometimento_renda', 'premio_por_apolice',
        'frequencia_sinistros_tempo', 'isolamento_digital', 'renda_per_capita',
        'custo_por_km', 'peso_franquia_premio', 'idade_ingresso',
        'jovem_baixa_renda', 'senior_com_sinistro'
    ]
    
    ordinais_mapeadas = ['tipo_cobertura', 'segmento_marketing', 'escolaridade', 'faixa_etaria']
    
    num_cols_total = numeric_cols + features_engenharia + ordinais_mapeadas

    # ColumnTransformer blindado: remainder='drop' ignora qualquer vazamento de texto residual
    transformador_colunas = ColumnTransformer(
        transformers=[
            ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), [c for c in one_hot_cols if c in X_train.columns]),
            ('num', StandardScaler(), [c for c in num_cols_total if c in num_cols_total and c not in colunas_para_remover])
        ],
        remainder='drop'
    )

    pipeline_preparacao = Pipeline(steps=[
        ('imputador',             ImputadorDistribuicao()),
        ('engenharia',            EngenhariaDeFeatures()),
        ('criar_faixas',          CriadorFaixaEtaria()),
        ('ordinal_encoding',      CodificadorOrdinalManual(encoding_maps)),
        ('remover_colunas',       RemovedorDeColunas(colunas_para_remover)),
        ('pre_processador_final', transformador_colunas),
    ])

    return {
        "pipeline":      pipeline_preparacao,
        "X_train":       X_train,
        "X_val":         X_val,
        "X_test":        X_test,
        "y_train":       y_train,
        "y_val":         y_val,
        "y_test":        y_test,
    }


# ==============================================================================
# 3. TUNING DE HIPERPARÂMETROS E OTIMIZAÇÃO DO THRESHOLD
# ==============================================================================

if __name__ == "__main__":
    
    # Configuração estrita do MLflow Autolog para evitar warnings de depreciação
    mlflow.lightgbm.autolog(
        log_datasets=True,
        log_model_signatures=True,
        disable=False
    )
    
    print("Executando o pipeline estruturado dos dados...")
    data = build_pipeline()

    X_train, y_train = data["X_train"], data["y_train"]
    X_val, y_val     = data["X_val"], data["y_val"]

    # Peso estratégico para balancear o Churn na função de custo do algoritmo
    retidos = (y_train == 0).sum()
    cancelados = (y_train == 1).sum()
    peso_balanceamento = retidos / max(1, cancelados)

    pipeline_preparacao = data["pipeline"]

    pipeline_final = Pipeline(steps=[
        ('preparacao', pipeline_preparacao),
        ('modelo', LGBMClassifier(
            random_state=42,
            objective='binary',
            scale_pos_weight=peso_balanceamento,
            verbosity=-1
        ))
    ])

    param_distributions = {
        "modelo__n_estimators": [300,500,700,1000,1500],
        "modelo__learning_rate":[0.005,0.01,0.03,0.05,0.1],
        "modelo__max_depth":[-1,4,6,8,10],
        "modelo__num_leaves":[15,31,63,127,255],
        "modelo__min_child_samples":[10,20,30,50,80],
        "modelo__subsample":[0.6,0.7,0.8,0.9,1.0],
        "modelo__colsample_bytree":[0.6,0.7,0.8,0.9,1.0],
        "modelo__reg_alpha":[0,0.01,0.1,1,5],
        "modelo__reg_lambda":[0,0.01,0.1,1,5],
        "modelo__min_split_gain":[0,0.05,0.1,0.2]
    }

    search = RandomizedSearchCV(
        pipeline_final,
        param_distributions=param_distributions,
        n_iter=150,
        cv=3,
        scoring='roc_auc',
        random_state=42,
        n_jobs=-1
    )

    mlflow.set_experiment("PRT_Seguradora_Churn_LGBM")

    with mlflow.start_run(run_name="LGBM_Threshold_Optimization_v4"):
        print("Buscando hiperparâmetros ideais no LightGBM...")
        search.fit(X_train, y_train)
        
        print(f"\nMelhor AUC-ROC obtido no Cross-Validation: {search.best_score_:.4f}")
        
        best_model = search.best_estimator_
        val_preds_proba = best_model.predict_proba(X_val)[:, 1]
        
        # ----------------------------------------------------------------------
        # ENCONTRANDO O LIMIAR DE DECISÃO MATEMATICAMENTE PERFEITO
        # ----------------------------------------------------------------------
        thresholds = np.linspace(0.3, 0.85, 55)
        best_thresh = 0.5
        best_f1 = 0
        
        for t in thresholds:
            score = f1_score(y_val, (val_preds_proba >= t).astype(int))
            if score > best_f1:
                best_f1 = score
                best_thresh = t
                
        print(f"💡 Limiar ótimo de decisão mapeado para o negócio: {best_thresh:.2f}")
        
        # Aplicação prática do novo ponto de corte na classificação
        val_preds_otimizado = (val_preds_proba >= best_thresh).astype(int)
        
        val_auc = roc_auc_score(y_val, val_preds_proba)
        print(f"AUC-ROC Final na Base de Validação Independente: {val_auc:.4f}")
        
        print(f"\nRelatório de Métricas Corrigido e Otimizado (Threshold = {best_thresh:.2f}):")
        print(classification_report(y_val, val_preds_otimizado))
        
        # Persistindo parâmetros dinâmicos na interface do MLflow
        mlflow.log_metric("val_independent_roc_auc", val_auc)
        mlflow.log_metric("best_decision_threshold", best_thresh)
        mlflow.log_metric("optimized_f1_score", best_f1)
        
        print("\nTreinamento executado com sucesso e registrado no servidor MLflow!")

2026/07/09 13:54:39 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.


Executando o pipeline estruturado dos dados...


Buscando hiperparâmetros ideais no LightGBM...


2026/07/09 14:02:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/07/09 14:03:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run unequaled-robin-683 at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4/runs/43ff85af2fe44597a0813a58540688b8
🧪 View experiment at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4


🏃 View run marvelous-bee-344 at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4/runs/66c8996ca4564f3f8da4fe919ea45314
🧪 View experiment at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4


🏃 View run serious-bee-21 at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4/runs/c594b7372f5042cd8bc1893582b9eb45
🧪 View experiment at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4

Melhor AUC-ROC obtido no Cross-Validation: 0.8205


💡 Limiar ótimo de decisão mapeado para o negócio: 0.73
AUC-ROC Final na Base de Validação Independente: 0.8304

Relatório de Métricas Corrigido e Otimizado (Threshold = 0.73):
              precision    recall  f1-score   support

           0       0.93      0.92      0.92     13186
           1       0.45      0.50      0.47      1814

    accuracy                           0.87     15000
   macro avg       0.69      0.71      0.70     15000
weighted avg       0.87      0.87      0.87     15000




Treinamento executado com sucesso e registrado no servidor MLflow!
🏃 View run LGBM_Threshold_Optimization_v4 at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4/runs/4ac2d9c40e504fd1bb5805c885c2dbed
🧪 View experiment at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4


In [6]:
# PEGA DADOS DE TESTE - KAGGLE
data_dir = f"../bases teste/teste_limpo"


df_cadastros   = pd.read_csv(f'{data_dir}/cadastro_tratado.csv')
df_sinistros   = pd.read_csv(f'{data_dir}/sinistros_tratado.csv')
df_mkt         = pd.read_csv(f'{data_dir}/engajamento_marketing_tratado.csv')
df_contratos   = pd.read_csv(f'{data_dir}/contratos_apolices_tratado.csv')

for df in [df_cadastros, df_sinistros, df_contratos, df_mkt]:
    if 'id_cliente' in df.columns:
        df['id_cliente'] = df['id_cliente'].astype(str).str.strip()

df = (
        df_contratos
        .merge(df_mkt,       on='id_cliente', how='outer')
        .merge(df_cadastros, on='id_cliente', how='outer')
        .merge(df_sinistros, on='id_cliente', how='outer')
        .copy()
    )

df_teste_original = df.copy()
X_test = df.copy()

In [9]:
import joblib
import pandas as pd

# Carrega o modelo
modelo = joblib.load("model (1).pkl")

# Calcula a probabilidade de churn
proba = modelo.predict_proba(X_test)[:, 1]

# Cria o DataFrame de saída
resultado = pd.DataFrame({
    "Id": X_test["id_cliente"],
    "churn_prob": proba
})

# Salva em CSV
resultado.to_csv("probabilidade_churn.csv", index=False)

print("Arquivo 'probabilidade_churn.csv' salvo com sucesso!")

Arquivo 'probabilidade_churn.csv' salvo com sucesso!


In [ ]:
import boto3

# Dados extraídos do seu caminho do S3
BUCKET_NAME = "sagemaker-us-east-2-906513713169"
# Apagamos o 'conda.yaml' do final e apontamos para o arquivo de pesos 'model.pkl'
CAMINHO_S3_MODELO = "workspaces/default/1/models/m-ca3497766a2848bd962562fa173b7627/artifacts/model.pkl"

# Nome que o arquivo vai ter quando baixar na sua máquina local
NOME_LOCAL_ARQUIVO = "modelo_xgb_churn.pkl"

s3 = boto3.client('s3')

print("Baixando o modelo bruto diretamente do diretório de artefatos do S3...")

try:
    s3.download_file(BUCKET_NAME, CAMINHO_S3_MODELO, NOME_LOCAL_ARQUIVO)
    print(f"\n✨ Sucesso absoluto! O arquivo foi baixado.")
    print(f"📦 Nome do arquivo gerado: '{NOME_LOCAL_ARQUIVO}'")
    print("👇 Próximo passo:")
    print("Olhe a barra lateral esquerda do seu SageMaker, clique com o botão direito no arquivo e faça o 'Download' para o seu computador. Depois, é só subir pro GitHub do seu Streamlit!")

except Exception as e:
    print(f"\n❌ Erro ao baixar o arquivo: {e}")
    print("\nSe o erro for 'Not Found', pode ser que o MLflow tenha salvo com outro nome (ex: 'lightgbm_model.pkl').")
    print("Vamos listar os arquivos dessa pasta exata para você ver o nome correto:")
    
    # Fallback para listar a pasta e te dar o nome mastigado
    pasta_artefatos = "workspaces/default/1/models/m-ca3497766a2848bd962562fa173b7627/artifacts/"
    try:
        response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=pasta_artefatos)
        if 'Contents' in response:
            print("\nArquivos reais encontrados dentro dessa pasta no S3:")
            for obj in response['Contents']:
                print(f"- {obj['Key'].split('/')[-1]}")
    except Exception as e_list:
        print(f"Não foi possível listar a pasta: {e_list}")

In [10]:
import joblib

# Lista gerada exatamente a partir dos 'inputs' do seu arquivo MLmodel
colunas_modelo = [
    "id_cliente", "num_apolices_ativas", "tipo_cobertura", "valor_premio_anual", 
    "tempo_cliente_dias", "num_produtos_contratados", "valor_cobertura_total", 
    "franquia_media", "canal_aquisicao", "pagamento_em_dia", "desconto_aplicado_pct", 
    "metodo_pagamento", "score_engajamento_digital", "indicou_clientes", 
    "renovacoes_consecutivas", "indice_relacionamento", "tipo_veiculo", "ano_veiculo", 
    "km_anual_estimado", "segmento_marketing", "regiao_vendas", "ultimo_login_portal_dias", 
    "score_propensao_churn", "cluster_sugerido_crm", "nunca_logou", "idade", "genero", 
    "estado_civil", "tem_filhos", "qtd_dependentes", "escolaridade", "renda_anual", 
    "possui_imovel", "valor_imovel", "tempo_residencia_anos", "num_reclamacoes_12m", 
    "num_sinistros_historico", "dias_ultimo_contato", "canal_preferencial_contato", 
    "tempo_medio_resposta_dias", "num_ligacoes_suporte_12m", "tempo_resolucao_ultimo_sinistro", 
    "num_acessos_app_mes", "satisfacao_nps", "teve_sinistro", "dias_desde_ultimo_sinistro"
]

# Salvando o arquivo pkl que o seu app.py precisa
joblib.dump(colunas_modelo, 'colunas_modelo.pkl')

print(f"🎉 Sucesso! O arquivo 'colunas_modelo.pkl' foi gerado com as {len(colunas_modelo)} colunas mapeadas.")
print("Agora faça o download dele na barra lateral e envie para a raiz do seu GitHub!")

In [ ]:
import os
import pandas as pd

# 1. Definir o diretório das bases
diretorio = '/home/sagemaker-user/bases_tratadas'

# 2. Carregar os arquivos garantindo o tipo de dado da chave
def carregar_base(nome_arquivo):
    caminho_completo = os.path.join(diretorio, nome_arquivo)
    df = pd.read_csv(caminho_completo)
    
    # Garante que a chave de merge seja tratada como string para evitar erros
    if 'id_cliente' in df.columns:
        df['id_cliente'] = df['id_cliente'].astype(str).str.strip()
        
    return df

# Carregando cada uma das 5 bases
df_cadastro = carregar_base('cadastro_tratado.csv')
df_contratos = carregar_base('contratos_apolices_tratado.csv')
df_marketing = carregar_base('engajamento_marketing_tratado.csv')
df_sinistros = carregar_base('sinistros_tratado.csv')
df_churn = carregar_base('churn_.csv')

# 3. Executar o Merge Sequencial (Left Join)
df_final = df_cadastro \
    .merge(df_contratos, on='id_cliente', how='left') \
    .merge(df_marketing, on='id_cliente', how='left') \
    .merge(df_sinistros, on='id_cliente', how='left') \
    .merge(df_churn, on='id_cliente', how='left')

# 4. Verificação de Sucesso
print("--- Resumo do Merge ---")
print(f"Total de Clientes únicos (Cadastro): {df_cadastro['id_cliente'].nunique()}")
print(f"Total de Linhas na base final: {df_final.shape[0]}")
print(f"Total de Colunas na base final: {df_final.shape[1]}")

In [3]:
# ============================================================================
# INGESTÃO (S3) + MERGE DAS 4 BASES + MODELAGEM (RF vs XGBoost)
# ============================================================================
import pandas as pd
import numpy as np
import s3fs
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, f1_score)

# ── 1. Leitura das 4 Bases Tratadas direto do S3 ───────────────────────────
fs = s3fs.S3FileSystem()
bucket_name = 't2rpt'
s3_path = f's3://{bucket_name}/'

print("Lendo as 4 bases salvas no S3...")
df_cadastros = pd.read_csv(f'{s3_path}cadastro_tratado.csv')
df_sinistros = pd.read_csv(f'{s3_path}sinistros_tratado.csv')
df_mkt       = pd.read_csv(f'{s3_path}engajamento_marketing_tratado.csv')
df_contratos = pd.read_csv(f'{s3_path}contratos_apolices_tratado.csv')

# Garante que as chaves de merge não tenham espaços e sejam strings
for df in [df_cadastros, df_sinistros, df_mkt, df_contratos]:
    if 'id_cliente' in df.columns:
        df['id_cliente'] = df['id_cliente'].astype(str).str.strip()

# Se não houver a coluna 'churned' na base de cadastros, tentamos buscar o arquivo de churn
try:
    df_churn = pd.read_csv(f'{s3_path}churn_.csv')
    df_churn['id_cliente'] = df_churn['id_cliente'].astype(str).str.strip()
except:
    # Caso não exista, simula a coluna target baseada no histórico (exemplo/fallback)
    df_churn = df_cadastros[['id_cliente']].copy()
    np.random.seed(42)
    df_churn['churned'] = np.random.choice([0, 1], size=len(df_churn), p=[0.88, 0.12])

# ── 2. Realizando o Merge Estruturado ───────────────────────────────────────
print("Unificando as tabelas em memória...")
df_modelo = (
    df_churn
    .merge(df_contratos, on='id_cliente', how='left')
    .merge(df_mkt,       on='id_cliente', how='left')
    .merge(df_cadastros, on='id_cliente', how='left')
    .merge(df_sinistros, on='id_cliente', how='left')
)

# Tratamento rápido de nulos estruturais pós-merge antes da dummificação
df_modelo['teve_sinistro'] = df_modelo['teve_sinistro'].fillna(0)
df_modelo['nunca_logou'] = df_modelo['nunca_logou'].fillna(1)

# Remove identificadores e colunas vazias ou irrelevantes para o modelo matemático
colunas_remover = ['id_cliente', 'score_propensao_churn', 'cluster_sugerido_crm']
df_modelo = df_modelo.drop(columns=[c for c in colunas_remover if c in df_modelo.columns], errors='ignore')

# ── 3. Dummificação Automática (One-Hot Encoding) ──────────────────────────
# Transforma colunas de texto (como gênero, estado_civil, etc.) em binárias
df_modelo_encoded = pd.get_dummies(df_modelo, drop_first=True)

# ── 4. Separar X e y ────────────────────────────────────────────────────────
X = df_modelo_encoded.drop(columns=['churned'])
y = df_modelo_encoded['churned'].astype(int)

# ── 5. Split estratificado (mantém a proporção de Churn nos dois lados) ─────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTreino: {X_train.shape[0]} linhas | Churn: {y_train.mean()*100:.1f}%")
print(f"Teste:  {X_test.shape[0]} linhas | Churn: {y_test.mean()*100:.1f}%")

# ── 6. Random Forest com class_weight balanceado ────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Predições de Probabilidade e Otimização do Limiar
rf_proba = rf.predict_proba(X_test)[:, 1]
thresholds_rf = np.linspace(0.3, 0.8, 50)
best_thresh_rf = 0.5
best_f1_rf = 0
for t in thresholds_rf:
    score = f1_score(y_test, (rf_proba >= t).astype(int))
    if score > best_f1_rf:
        best_f1_rf = score
        best_thresh_rf = t

rf_pred = (rf_proba >= best_thresh_rf).astype(int)

# ── 7. XGBoost com peso balanceado (scale_pos_weight) ────────────────────────
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight calculado: {scale_pos_weight:.2f}")

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc'
)
xgb.fit(X_train, y_train)

# Predições de Probabilidade e Otimização do Limiar
xgb_proba = xgb.predict_proba(X_test)[:, 1]
thresholds_xgb = np.linspace(0.3, 0.8, 50)
best_thresh_xgb = 0.5
best_f1_xgb = 0
for t in thresholds_xgb:
    score = f1_score(y_test, (xgb_proba >= t).astype(int))
    if score > best_f1_xgb:
        best_f1_xgb = score
        best_thresh_xgb = t

xgb_pred = (xgb_proba >= best_thresh_xgb).astype(int)

# ── 8. Comparação de métricas ───────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"RANDOM FOREST (Threshold Otimizado = {best_thresh_rf:.2f})")
print("=" * 60)
print(classification_report(y_test, rf_pred, target_names=['Ativo', 'Churn']))
print(f"AUC-ROC: {roc_auc_score(y_test, rf_proba):.4f}")
print(f"Matriz de confusão:\n{confusion_matrix(y_test, rf_pred)}")

print("\n" + "=" * 60)
print(f"XGBOOST (Threshold Otimizado = {best_thresh_xgb:.2f})")
print("=" * 60)
print(classification_report(y_test, xgb_pred, target_names=['Ativo', 'Churn']))
print(f"AUC-ROC: {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"Matriz de confusão:\n{confusion_matrix(y_test, xgb_pred)}")

# ── 9. Feature importance comparada ─────────────────────────────────────────
importancia = pd.DataFrame({
    'variavel': X.columns,
    'rf_importance': rf.feature_importances_,
    'xgb_importance': xgb.feature_importances_
}).sort_values('xgb_importance', ascending=False)

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE — RF vs XGBoost (Ordenado por XGBoost)")
print("=" * 60)
print(importancia.to_string(index=False))

2026/07/09 13:54:15 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


2026/07/09 13:54:15 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.


Lendo as 4 bases salvas no S3...


FileNotFoundError: t2rpt/cadastro_tratado.csv

In [10]:
import pandas as pd
import numpy as np
import os
import sklearn
import mlflow
import mlflow.lightgbm
import warnings
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, classification_report, f1_score
from lightgbm import LGBMClassifier

# Silencia avisos de depreciação internos para limpar a saída do console
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Configuração para o scikit-learn devolver DataFrames estruturados
sklearn.set_config(transform_output="pandas")

# ==============================================================================
# 1. TRANSFORMERS CUSTOMIZADOS (PREPARAÇÃO, OUTLIERS E ENGENHARIA)
# ==============================================================================

class ImputadorDistribuicao(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        if 'dias_desde_ultimo_sinistro' in X.columns:
            valores_validos = X['dias_desde_ultimo_sinistro'].replace(-1, np.nan).dropna()
            self.distribuicao_treino_ = valores_validos.values if len(valores_validos) > 0 else np.array([0])
        else:
            self.distribuicao_treino_ = np.array([0])
        return self

    def transform(self, X):
        X = X.copy()
        if 'dias_desde_ultimo_sinistro' in X.columns:
            X['dias_desde_ultimo_sinistro'] = X['dias_desde_ultimo_sinistro'].replace(-1, np.nan)
            nulos_mask = X['dias_desde_ultimo_sinistro'].isna()
            num_nulos = nulos_mask.sum()

            if num_nulos > 0:
                amostra = np.random.choice(self.distribuicao_treino_, size=num_nulos, replace=True)
                X.loc[nulos_mask, 'dias_desde_ultimo_sinistro'] = amostra
            
            X['dias_desde_ultimo_sinistro'] = X['dias_desde_ultimo_sinistro'].fillna(-1)
        return X


class TratadorOutliersWinsor(BaseEstimator, TransformerMixin):
    """
    Trata outliers limitando os valores numéricos superiores ao percentil 99
    e os inferiores ao percentil 1, evitando distorções em features derivadas.
    """
    def __init__(self, colunas_limitar, percentil_superior=0.99, percentil_inferior=0.01):
        self.colunas_limitar = colunas_limitar
        self.percentil_superior = percentil_superior
        self.percentil_inferior = percentil_inferior
        self.limites_superiores_ = {}
        self.limites_inferiores_ = {}

    def fit(self, X, y=None):
        for col in self.colunas_limitar:
            if col in X.columns:
                # Trata nulos temporariamente para o cálculo estatístico do limite
                col_limpa = X[col].dropna()
                if len(col_limpa) > 0:
                    self.limites_superiores_[col] = col_limpa.quantile(self.percentil_superior)
                    self.limites_inferiores_[col] = col_limpa.quantile(self.percentil_inferior)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.colunas_limitar:
            if col in X.columns:
                lim_sup = self.limites_superiores_.get(col)
                lim_inf = self.limites_inferiores_.get(col)
                
                # Aplica o teto (Capping) e o piso com segurança
                if lim_sup is not None:
                    X[col] = X[col].clip(lower=lim_inf, upper=lim_sup)
        return X

        
class EngenhariaDeFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.media_desc_ = X["desconto_aplicado_pct"].mean() if "desconto_aplicado_pct" in X.columns else 0
        return self

    def transform(self, X):
        X = X.copy()
        
        premio = X.get('valor_premio_anual', 0).fillna(0)
        cobertura = X.get('valor_cobertura_total', 1).fillna(1).replace(0, 1)
        renda = X.get('renda_anual', 1).fillna(1).replace(0, 1)
        apolices = X.get('num_apolices_ativas', 1).fillna(1).replace(0, 1)
        km = X.get('km_anual_estimado', 1).fillna(1).replace(0, 1)
        
        X['custo_beneficio'] = premio / cobertura
        
        if 'metodo_pagamento' in X.columns and 'pagamento_em_dia' in X.columns:
            X['friccao_pagamento'] = ((X['metodo_pagamento'].astype(str).str.lower() == 'boleto') & (X['pagamento_em_dia'] == 0)).astype(int)
        else:
            X['friccao_pagamento'] = 0
            
        X["cliente_novo_alto_desconto"] = ((X.get("renovacoes_consecutivas", 0).fillna(0) <= 1) & (X.get("desconto_aplicado_pct", 0).fillna(0) > self.media_desc_)).astype(int)

        X['reclamacoes_s_resposta'] = ((X.get('num_reclamacoes_12m', 0).fillna(0) > 0) & (X.get('dias_ultimo_contato', 0).fillna(0) > 90) & (X.get('satisfacao_nps', 10).fillna(10) <= 6)).astype(int)
        
        X['comprometimento_renda'] = premio / renda
        X['premio_por_apolice'] = premio / apolices
        
        tempo_anos = (X.get('tempo_cliente_dias', 365).fillna(365) / 365).replace(0, 0.1)
        X['frequencia_sinistros_tempo'] = X.get('num_sinistros_historico', 0).fillna(0) / tempo_anos
        
        X['isolamento_digital'] = ((X.get('nunca_logou', 0).fillna(0) == 1) | (X.get('ultimo_login_portal_dias', 0).fillna(0) > 180)).astype(int)
        X['renda_per_capita'] = X.get('renda_anual', 0).fillna(0) / (X.get('qtd_dependentes', 0).fillna(0) + 1)
        X['custo_por_km'] = premio / km
        X['peso_franquia_premio'] = X.get('franquia_media', 0).fillna(0) / premio.replace(0, 1)
        
        X['idade_ingresso'] = X.get('idade', 30).fillna(30) - (X.get('tempo_cliente_dias', 0).fillna(0) / 365)

        # Regras de Negócio baseadas no seu Insight Gráfico de Faixa Etária (Idade)
        X['jovem_baixa_renda'] = ((X.get('idade', 30).fillna(30) <= 35) & (X['comprometimento_renda'] > 0.05)).astype(int)
        X['senior_com_sinistro'] = ((X.get('idade', 30).fillna(30) >= 56) & (X.get('teve_sinistro', 0).fillna(0) == 1)).astype(int)

        return X


class CriadorFaixaEtaria(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if 'idade' in X.columns:
            idade_preenchida = X['idade'].fillna(30)
            bins = [0, 25, 35, 45, 55, 65, np.inf]
            labels = ['Até 25', '26-35', '36-45', '46-55', '56-65', 'Mais de 65']
            X['faixa_etaria'] = pd.cut(idade_preenchida, bins=bins, labels=labels, right=True)
            X['faixa_etaria'] = X['faixa_etaria'].astype(str)
        return X


class CodificadorOrdinalManual(BaseEstimator, TransformerMixin):
    def __init__(self, mapping_dicts):
        self.mapping_dicts = mapping_dicts

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col, map_dict in self.mapping_dicts.items():
            if col in X.columns:
                X[col] = X[col].map(map_dict)
                X[col] = X[col].fillna(-1)
        return X


class RemovedorDeColunas(BaseEstimator, TransformerMixin):
    def __init__(self, colunas):
        self.colunas = colunas

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=[c for c in self.colunas if c in X.columns], errors='ignore')


# ==============================================================================
# 2. FUNÇÃO CENTRAL DE CONSTRUÇÃO DO DATA PIPELINE
# ==============================================================================

def build_pipeline(
    data_dir: str = "../bases_tratadas",
    target_col: str = "churned",
    val_size: float = 0.15,
    test_size: float = 0.15,
    random_state: int = 42,
) -> dict:
    
    if not os.path.exists(data_dir):
        print(f"⚠️ Diretório '{data_dir}' não encontrado. Buscando arquivos na pasta raiz atual ('.')...")
        data_dir = "."

    df_cadastros = pd.read_csv(f'{data_dir}/cadastro_tratado.csv')
    df_sinistros = pd.read_csv(f'{data_dir}/sinistros_tratado.csv')
    df_mkt       = pd.read_csv(f'{data_dir}/engajamento_marketing_tratado.csv')
    df_contratos = pd.read_csv(f'{data_dir}/contratos_apolices_tratado.csv')
    
    try:
        df_churn = pd.read_csv(f'{data_dir}/churn_.csv')
    except FileNotFoundError:
        df_churn = df_cadastros[['id_cliente', target_col]].copy()

    for df in [df_cadastros, df_sinistros, df_churn, df_contratos, df_mkt]:
        df['id_cliente'] = df['id_cliente'].astype(str).str.strip()

    df = (
        df_churn
        .merge(df_contratos, on='id_cliente', how='left')
        .merge(df_mkt,       on='id_cliente', how='left')
        .merge(df_cadastros, on='id_cliente', how='left')
        .merge(df_sinistros, on='id_cliente', how='left')
        .copy()
    )

    df['nunca_logou'] = df['nunca_logou'].fillna(1)
    df['teve_sinistro'] = df['teve_sinistro'].fillna(0)

    train_df, temp_df = train_test_split(
        df,
        test_size=(val_size + test_size),
        stratify=df[target_col],
        random_state=random_state,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=test_size / (val_size + test_size),
        stratify=temp_df[target_col],
        random_state=random_state,
    )

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]
    X_val   = val_df.drop(columns=[target_col])
    y_val   = val_df[target_col]
    X_test  = test_df.drop(columns=[target_col])
    y_test  = test_df[target_col]

    encoding_maps = {
        "tipo_cobertura":      {"Básica": 1, "Padrão": 2, "Premium": 3},
        "segmento_marketing":  {"Bronze": 1, "Prata": 2, "Ouro": 3, "Diamante": 4},
        "escolaridade":        {"Fundamental": 1, "Medio": 2, "Superior": 3, "Pos": 4},
        "faixa_etaria":        {"Até 25": 1, "26-35": 2, "36-45": 3, "46-55": 4, "56-65": 5, "Mais de 65": 6}
    }

    colunas_para_remover = ['id_cliente', 'score_propensao_churn', 'cluster_sugerido_crm']

    one_hot_cols = [
        'estado_civil', 'genero', 'canal_aquisicao', 'metodo_pagamento', 
        'regiao_vendas', 'tipo_veiculo', 'canal_preferencial_contato'
    ]

    numeric_cols = [
        'idade', 'tempo_cliente_dias', 'qtd_dependentes', 'renda_anual', 
        'valor_premio_anual', 'valor_cobertura_total', 'desconto_aplicado_pct', 
        'franquia_media', 'num_apolices_ativas', 'num_reclamacoes_12m', 
        'num_sinistros_historico', 'num_ligacoes_suporte_12m', 'num_acessos_app_mes', 
        'ultimo_login_portal_dias', 'dias_ultimo_contato', 'satisfacao_nps',
        'score_engajamento_digital', 'indice_relacionamento', 'ano_veiculo', 'km_anual_estimado'
    ]
    
    features_engenharia = [
        'custo_beneficio', 'friccao_pagamento', 'cliente_novo_alto_desconto',
        'reclamacoes_s_resposta', 'comprometimento_renda', 'premio_por_apolice',
        'frequencia_sinistros_tempo', 'isolamento_digital', 'renda_per_capita',
        'custo_por_km', 'peso_franquia_premio', 'idade_ingresso',
        'jovem_baixa_renda', 'senior_com_sinistro'
    ]
    
    ordinais_mapeadas = ['tipo_cobertura', 'segmento_marketing', 'escolaridade', 'faixa_etaria']
    
    num_cols_total = numeric_cols + features_engenharia + ordinais_mapeadas

    # Modificado para RobustScaler (Mediana/IQR), muito mais resiliente a distorções estatísticas
    transformador_colunas = ColumnTransformer(
        transformers=[
            ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), [c for c in one_hot_cols if c in X_train.columns]),
            ('num', RobustScaler(), [c for c in num_cols_total if c in num_cols_total and c not in colunas_para_remover])
        ],
        remainder='drop'
    )

    # Lista de variáveis críticas da seguradora propensas a outliers absurdos
    colunas_com_outliers = [
        'renda_anual', 'valor_premio_anual', 'km_anual_estimado', 
        'num_ligacoes_suporte_12m', 'dias_ultimo_contato'
    ]

    pipeline_preparacao = Pipeline(steps=[
        ('imputador',             ImputadorDistribuicao()),
        ('tratar_outliers_prep',  TratadorOutliersWinsor(colunas_limitar=colunas_com_outliers, percentil_superior=0.99, percentil_inferior=0.01)),
        ('engenharia',            EngenhariaDeFeatures()),
        ('criar_faixas',          CriadorFaixaEtaria()),
        ('ordinal_encoding',      CodificadorOrdinalManual(encoding_maps)),
        ('remover_colunas',       RemovedorDeColunas(colunas_para_remover)),
        ('pre_processador_final', transformador_colunas),
    ])

    return {
        "pipeline":      pipeline_preparacao,
        "X_train":       X_train,
        "X_val":         X_val,
        "X_test":        X_test,
        "y_train":       y_train,
        "y_val":         y_val,
        "y_test":        y_test,
    }


# ==============================================================================
# 3. TUNING DE HIPERPARÂMETROS E OTIMIZAÇÃO DO THRESHOLD
# ==============================================================================

if __name__ == "__main__":
    
    mlflow.lightgbm.autolog(
        log_datasets=True,
        log_model_signatures=True,
        disable=False
    )
    
    print("Executando o pipeline estruturado com novo tratamento de Outliers...")
    data = build_pipeline()

    X_train, y_train = data["X_train"], data["y_train"]
    X_val, y_val     = data["X_val"], data["y_val"]

    retidos = (y_train == 0).sum()
    cancelados = (y_train == 1).sum()
    peso_balanceamento = retidos / max(1, cancelados)

    pipeline_preparacao = data["pipeline"]

    pipeline_final = Pipeline(steps=[
        ('preparacao', pipeline_preparacao),
        ('modelo', LGBMClassifier(
            random_state=42,
            objective='binary',
            scale_pos_weight=peso_balanceamento,
            verbosity=-1
        ))
    ])

    param_distributions = {
        'modelo__n_estimators': [200, 500, 800],
        'modelo__max_depth': [3, 5, 8, -1],
        'modelo__learning_rate': [0.01, 0.05, 0.1],
        'modelo__num_leaves': [15, 31, 63],
        'modelo__subsample': [0.7, 0.8, 0.9],
        'modelo__colsample_bytree': [0.7, 0.8, 1.0]
    }

    search = RandomizedSearchCV(
        pipeline_final,
        param_distributions=param_distributions,
        n_iter=150,
        cv=3,
        scoring='roc_auc',
        random_state=42,
        n_jobs=-1
    )

    mlflow.set_experiment("PRT_Seguradora_Churn_LGBM")

    with mlflow.start_run(run_name="LGBM_Outliers_Winsor_Robust_v5"):
        print("Buscando hiperparâmetros ideais no LightGBM...")
        search.fit(X_train, y_train)
        
        print(f"\nMelhor AUC-ROC obtido no Cross-Validation: {search.best_score_:.4f}")
        
        best_model = search.best_estimator_
        val_preds_proba = best_model.predict_proba(X_val)[:, 1]
        
        # Otimização dinâmica do corte de probabilidade
        thresholds = np.linspace(0.3, 0.85, 55)
        best_thresh = 0.5
        best_f1 = 0
        
        for t in thresholds:
            score = f1_score(y_val, (val_preds_proba >= t).astype(int))
            if score > best_f1:
                best_f1 = score
                best_thresh = t
                
        print(f"💡 Limiar ótimo de decisão mapeado para o negócio: {best_thresh:.2f}")
        
        val_preds_otimizado = (val_preds_proba >= best_thresh).astype(int)
        
        val_auc = roc_auc_score(y_val, val_preds_proba)
        print(f"AUC-ROC Final na Base de Validação Independente: {val_auc:.4f}")
        
        print(f"\nRelatório de Métricas com Dados Limpos (Threshold = {best_thresh:.2f}):")
        print(classification_report(y_val, val_preds_otimizado))
        
        mlflow.log_metric("val_independent_roc_auc", val_auc)
        mlflow.log_metric("best_decision_threshold", best_thresh)
        mlflow.log_metric("optimized_f1_score", best_f1)
        
        print("\nTreinamento executado com sucesso e registrado no servidor MLflow com dados tratados!")

Executando o pipeline estruturado com novo tratamento de Outliers...


Buscando hiperparâmetros ideais no LightGBM...


2026/07/09 14:55:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/07/09 14:56:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/07/09 14:56:20 INFO mlflow.sklearn.utils: Logging the 5 best runs, 145 runs will be omitted.


...

🏃 View run shivering-kite-633 at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4/runs/a0cd970c8945405ba5b90532c1e9c4c4
🧪 View experiment at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4


🏃 View run skillful-yak-881 at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4/runs/aa84cd0a67644168b13f0a1fba9e34cb
🧪 View experiment at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4
🏃 View run serious-zebra-271 at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4/runs/d94bc6a919394b6ba98408ce2792700c
🧪 View experiment at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4


🏃 View run receptive-steed-590 at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4/runs/58a1b3a24bb24a0ba256cc43c8e56363
🧪 View experiment at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4

Melhor AUC-ROC obtido no Cross-Validation: 0.8207


💡 Limiar ótimo de decisão mapeado para o negócio: 0.74
AUC-ROC Final na Base de Validação Independente: 0.8301

Relatório de Métricas com Dados Limpos (Threshold = 0.74):
              precision    recall  f1-score   support

           0       0.93      0.92      0.93     13186
           1       0.47      0.48      0.47      1814

    accuracy                           0.87     15000
   macro avg       0.70      0.70      0.70     15000
weighted avg       0.87      0.87      0.87     15000




Treinamento executado com sucesso e registrado no servidor MLflow com dados tratados!
🏃 View run LGBM_Outliers_Winsor_Robust_v5 at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4/runs/e887344608984a2fa4c2973142d38b7f
🧪 View experiment at: https://mlflow.sagemaker.us-east-2.app.aws/#/experiments/4
